# Pattern 1: Prompt chaining

**Idea:** split one big task into a fixed sequence of small steps. Each step is a separate model call, and the output of step N becomes the input of step N+1. Between steps, plain Python code can check the result (a "gate") and stop early.

```
raw input ──▶ [LLM: extract] ──▶ gate ──▶ [LLM: prioritize] ──▶ [LLM: draft reply] ──▶ output
```

**Why it exists:** one giant prompt ("read this, find issues, rank them, write a reply") asks the model to do several jobs at once, and when it goes wrong you can't tell which job failed. Small steps are easier for the model and much easier for you to debug, test, and log.

**Trade-off:** more calls means more latency and cost. You trade speed for accuracy and control.

**Java analogy:** a Spring Batch job with ordered steps, or a Unix pipe `cat | grep | sort`. Each stage does one thing; the sequence is decided by **your code**, not by the model.

**Scenario in this notebook:** turn a messy customer email into (1) a list of issues, (2) a prioritized list, (3) a polite reply.

## Step 0: Setup (same in every notebook)

Every notebook starts with the same setup cells, so you can open any of them alone without running the others first.

1. **Load configuration** from the `.env` file in the repo root. The variable names are the ones the Anthropic SDK and Claude Code already use (`ANTHROPIC_BASE_URL`, `ANTHROPIC_API_KEY` or `ANTHROPIC_AUTH_TOKEN`, `ANTHROPIC_MODEL`), so the same `.env` can serve both.
2. **Create the client** with the official `anthropic` Python SDK. It sends requests to `ANTHROPIC_BASE_URL`: Anthropic's own API, or any gateway that exposes the Anthropic Messages API.
3. **Define a helper**, `call_claude()`, that wraps one API call and prints everything: the prompt going in, the text coming out, token usage, and timing.

> **Java analogy:** `.env` + `python-dotenv` is like `application.properties` + Spring's `@Value`. The client is a `RestTemplate`/`WebClient` bean configured with a base URL. Swapping the endpoint is a config change, not a code change, just like pointing a JDBC URL at a different database that speaks the same protocol.

In [ ]:
# --- Cell 0.1: load configuration from .env ---------------------------------
import os
import json
import time
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) walks UP from the notebook's folder until it finds a .env file.
# That's why the .env can live in the repo root while notebooks live in notebooks/.
env_path = find_dotenv(usecwd=True)
print(f"[setup] .env file found at: {env_path or 'NOT FOUND (using existing environment variables)'}")
# override=False: variables already set in your shell win over the file.
load_dotenv(env_path, override=False)

BASE_URL = os.getenv("ANTHROPIC_BASE_URL", "https://api.anthropic.com")
MODEL = os.getenv("ANTHROPIC_MODEL")
# Direct Anthropic access uses ANTHROPIC_API_KEY (sent as x-api-key).
# Gateways usually expect ANTHROPIC_AUTH_TOKEN (sent as Authorization: Bearer ...).
API_KEY = os.getenv("ANTHROPIC_API_KEY")
AUTH_TOKEN = os.getenv("ANTHROPIC_AUTH_TOKEN")

# Fail fast with a clear message instead of a confusing HTTP 401 later.
assert API_KEY or AUTH_TOKEN, "Set ANTHROPIC_API_KEY or ANTHROPIC_AUTH_TOKEN in .env (see .env.example)."
assert MODEL, "ANTHROPIC_MODEL is missing. Example: claude-sonnet-4-5"

def mask(secret):
    """Never print a full secret. Show just enough to confirm which one is loaded."""
    return None if not secret else secret[:4] + "..." + secret[-4:]

print(f"[setup] ANTHROPIC_BASE_URL   = {BASE_URL}")
print(f"[setup] ANTHROPIC_API_KEY    = {mask(API_KEY)}")
print(f"[setup] ANTHROPIC_AUTH_TOKEN = {mask(AUTH_TOKEN)}")
print(f"[setup] ANTHROPIC_MODEL      = {MODEL}")

In [ ]:
# --- Cell 0.2: create the Anthropic client -----------------------------------
import anthropic

# Optional: extra HTTP headers, one "Name: value" per line.
# Leave this unset when calling Anthropic directly. Set it only if your gateway
# needs headers of its own (for example, routing or tenant identification).
# Same variable name and format that Claude Code uses.
extra_headers = {}
for line in os.getenv("ANTHROPIC_CUSTOM_HEADERS", "").splitlines():
    if ":" in line:
        name, value = line.split(":", 1)
        extra_headers[name.strip()] = value.strip()
print(f"[setup] custom headers: {[f'{k}: {mask(v)}' for k, v in extra_headers.items()] or 'none'}")

client = anthropic.Anthropic(
    api_key=API_KEY,
    auth_token=AUTH_TOKEN,
    base_url=BASE_URL,
    default_headers=extra_headers,
)
print(f"[setup] Anthropic SDK version: {anthropic.__version__}")
print(f"[setup] Client ready. Every request goes to {BASE_URL}/v1/messages")

In [ ]:
# --- Cell 0.3: a logging helper around ONE model call -------------------------
# Everything in these notebooks is built from this single building block.
# The patterns are not new API features: they are just different ways of
# arranging plain calls like this one with ordinary Python code.

def call_claude(prompt, system=None, label="call", max_tokens=1024, verbose=True):
    """Send one user message to Claude and return the response text.

    prompt     -> the user message (str)
    system     -> optional system prompt: the model's role and rules
    label      -> a name for this call so the logs are easy to follow
    verbose    -> print the full prompt and response (set False inside threads)
    """
    if verbose:
        print("=" * 78)
        print(f"[{label}] >>> REQUEST")
        if system:
            print(f"[{label}] system: {system}")
        print(f"[{label}] user:\n{prompt}")

    started = time.perf_counter()
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    elapsed = time.perf_counter() - started

    # response.content is a LIST of blocks. For plain text calls there is one text block.
    text = "".join(block.text for block in response.content if block.type == "text")

    if verbose:
        print(f"[{label}] <<< RESPONSE ({elapsed:.2f}s, "
              f"in={response.usage.input_tokens} tok, out={response.usage.output_tokens} tok, "
              f"stop_reason={response.stop_reason})")
        print(text)
        print("=" * 78)
    if response.stop_reason == "max_tokens":
        print(f"[{label}] WARNING: response was cut off by max_tokens")
    return text


def parse_json(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

print("[setup] call_claude() and parse_json() are defined")

In [ ]:
# --- Cell 0.4: smoke test. If this works, your .env and gateway are fine -------
call_claude("Reply with exactly: gateway OK", label="smoke-test", max_tokens=20)

## Step 1: The input

A realistic, messy customer email. Nothing special here, just a Python string.

In [ ]:
customer_email = """
Hi, I ordered a laptop (order #A-1042) two weeks ago. It arrived with a cracked screen.
Also, I was charged twice on my credit card. And your support chat kept disconnecting me.
Honestly the packaging was nice though. Please fix this ASAP, I need it for work.
"""
print(f"[input] {len(customer_email)} characters")
print(customer_email)

## Step 2: Chain link 1, extract issues as JSON

We ask for **structured output** (JSON) so the next step, and our gate, can read it with code instead of guessing at prose.

In [ ]:
step1_system = ("You extract customer issues from support emails. "
                "Respond ONLY with a JSON array of short strings, one per distinct problem. "
                "Ignore compliments. No prose.")

step1_raw = call_claude(customer_email, system=step1_system, label="step1-extract")
issues = parse_json(step1_raw)

print(f"[step1] parsed {len(issues)} issues into a Python list:")
for i, issue in enumerate(issues, 1):
    print(f"   {i}. {issue}")

## Step 3: The gate (plain Python, no model involved)

This is the part people skip, and it's the most important. Between links we **validate in code**. If the email had no real issues there's nothing to prioritize, so we stop the chain and save two calls.

> Gates are deterministic. A model can be asked to "please stop if there are no issues," but `if len(issues) == 0` will *always* work.

In [ ]:
print(f"[gate] type check: issues is a {type(issues).__name__}")
assert isinstance(issues, list), "Step 1 did not return a JSON list"

if len(issues) == 0:
    print("[gate] No issues found. Stopping the chain here.")
    raise SystemExit("chain stopped by gate")
print(f"[gate] PASS: {len(issues)} issues, continuing to step 2")

## Step 4: Chain link 2, prioritize

Notice the input to this call is **only the output of step 1**, not the original email. Each link sees exactly what it needs.

In [ ]:
step2_system = ("You are a support triage lead. Given a JSON list of issues, return ONLY a JSON array "
                "of objects with keys 'issue' and 'priority' (one of: high, medium, low), "
                "sorted from highest to lowest priority.")

step2_input = json.dumps(issues, indent=2)
step2_raw = call_claude(step2_input, system=step2_system, label="step2-prioritize")
prioritized = parse_json(step2_raw)

print("[step2] prioritized issues:")
for item in prioritized:
    print(f"   [{item['priority'].upper():6}] {item['issue']}")

## Step 5: Chain link 3, draft the reply

The last link gets the prioritized list and writes for a human. Free text is fine here because nothing downstream parses it.

In [ ]:
step3_system = ("You write short, warm, professional customer support replies. "
                "Address every issue in priority order. Max 120 words. Sign as 'The Support Team'.")

step3_input = "Prioritized issues:\n" + json.dumps(prioritized, indent=2)
reply = call_claude(step3_input, system=step3_system, label="step3-reply")

## Recap

| Link | Input | Output | Who decides what's next? |
|---|---|---|---|
| 1. extract | raw email | JSON list | your code (gate) |
| 2. prioritize | JSON list | JSON list with priority | your code |
| 3. reply | prioritized list | text | end |

The **control flow lives in Python**. The model never decides which step runs next. That's the difference from an *agent* (see notebook 05), where the model chooses its own next action.

## Experiments to try

1. Replace `customer_email` with `"Thanks, everything was perfect!"` and watch the gate stop the chain.
2. Remove "Respond ONLY with a JSON array" from `step1_system`. Does `parse_json` still work? That's why structured output matters between links.
3. Collapse all three steps into one prompt and compare the quality and how hard it is to debug.